# TLC: Fine-tune Gemma 4 E4B for Hunter + Christine

Trains a QLoRA adapter on Colab T4 (16 GB) using SFT data generated by
`training/generate_training_data.ts`. One adapter per persona.

Inputs you provide:
- `data/sft_hunter.jsonl` and `data/sft_christine.jsonl` from the data-gen run
- A HF token (read access is fine; need to accept Gemma 4 license once)

Outputs:
- `out/lora-hunter/` and `out/lora-christine/` LoRA adapters
- (Optional) HF Hub upload for download to your local box

Wall-clock estimate: ~2 hours per persona on T4 with QLoRA + ~250 examples × 3 epochs.

## 1. Setup

Make sure the runtime is **T4 GPU** (Runtime → Change runtime type → T4 GPU).

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers==4.46.3 trl==0.12.2 peft==0.13.2 accelerate==1.1.1 bitsandbytes==0.44.1 datasets==3.1.0

In [ ]:
from huggingface_hub import login
import os

# Paste your HF token (Read-scoped is fine).
# Get one at https://huggingface.co/settings/tokens
HF_TOKEN = os.environ.get('HF_TOKEN') or input('HF token: ')
login(token=HF_TOKEN)

## 2. Upload your JSONL training data

Two ways:

**Easier**: drag-and-drop `sft_hunter.jsonl` and `sft_christine.jsonl` into the Files panel on the left.

**Cleaner**: push the data to a private HF dataset and `datasets.load_dataset` it. The cell below assumes drag-and-drop into `/content/`.


In [ ]:
import os
for f in ['sft_hunter.jsonl', 'sft_christine.jsonl']:
    p = f'/content/{f}'
    if not os.path.exists(p):
        raise FileNotFoundError(f'{p} not found. Drag {f} into the Files panel.')
    n = sum(1 for _ in open(p))
    print(f'{f}: {n} examples')

## 3. Load Gemma 4 E4B base + tokenizer (4-bit QLoRA)

We load the base model in 4-bit NF4 quantization, then attach a LoRA adapter
that trains in bf16. This is the standard QLoRA recipe — fits T4 with room to spare.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE = 'google/gemma-4-e4b-it'  # if HF blocks, use 'unsloth/gemma-4-E4B-it' as a mirror

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='eager',
)
model.config.use_cache = False  # required for grad-checkpoint + LoRA training

## 4. Configure LoRA

`r=16, alpha=32` is the standard balance; targeting all attention + MLP
projections gives the LoRA enough surface area to learn the strict-schema
behavior without going overboard.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

## 5. Train Hunter

~2 hours on T4 with ~250 examples × 3 epochs. The cell below trains in place;
if you stop it mid-run, restart from the `out/lora-hunter` checkpoint with
`SFTConfig(resume_from_checkpoint=...)`.

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

ds_hunter = load_dataset('json', data_files='/content/sft_hunter.jsonl', split='train')
print('hunter examples:', len(ds_hunter))

# Gemma's chat template uses "user"/"model" only (no "system", no "assistant").
# Fold system into the first user turn, rename assistant→model, then render
# to a text column the SFTTrainer can train on directly.
def _prep(ex):
    msgs = ex['messages']
    sys_txt = ''
    out = []
    for m in msgs:
        if m['role'] == 'system':
            sys_txt = m['content'].strip()
        elif m['role'] == 'user':
            content = (sys_txt + '\n\n' + m['content']) if sys_txt else m['content']
            out.append({'role': 'user', 'content': content})
            sys_txt = ''
        elif m['role'] == 'assistant':
            out.append({'role': 'model', 'content': m['content']})
        else:
            out.append(m)
    return {'text': tokenizer.apply_chat_template(out, tokenize=False, add_generation_prompt=False)}

ds_hunter = ds_hunter.map(_prep, remove_columns=ds_hunter.column_names)
print('rendered sample (first 400 chars):')
print(ds_hunter[0]['text'][:400])

cfg = SFTConfig(
    output_dir='out/lora-hunter',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    optim='paged_adamw_8bit',
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    max_length=4096,
    packing=True,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds_hunter,
    peft_config=peft_config,
    args=cfg,
)

trainer.train()
trainer.save_model('out/lora-hunter')

## 6. Train Christine

Reload the base model fresh (the previous train mutated it), then attach a
new LoRA. Same hyperparameters.

In [ ]:
import gc
del trainer, model
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='eager',
)
model.config.use_cache = False

ds_christine = load_dataset('json', data_files='/content/sft_christine.jsonl', split='train')
print('christine examples:', len(ds_christine))
ds_christine = ds_christine.map(_prep, remove_columns=ds_christine.column_names)

cfg.output_dir = 'out/lora-christine'

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds_christine,
    peft_config=peft_config,
    args=cfg,
)

trainer.train()
trainer.save_model('out/lora-christine')

## 7. Quick sanity check

Generate from the freshly-trained Christine adapter on a held-out topic. We're
not optimizing for output quality here — just want to confirm the adapter
loaded and the model emits a JSON-ish structure that mentions title /
objective / lesson_steps. Real schema validation happens back in TLC against
PersonaScaffoldSchema.

In [ ]:
from peft import PeftModel

test_prompt = '''Topic: Phases of the Moon
Grade level: 5th grade
Class length: 45 minutes
Subject: Science

No teacher-provided source material. Use your general knowledge; label all sections with source_origin="not_applicable" (where appropriate) or "generated".'''

messages = [
    {'role': 'user', 'content': test_prompt},
]
inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to(model.device)
out = model.generate(inputs, max_new_tokens=2048, do_sample=False)
decoded = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print(decoded[:2000])

## 8. Download the adapters

Two paths:

**A) Push to HF Hub** (recommended — reproducible + survives Colab disconnects):

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

HF_USER = 'hardcoded74'  # change if your HF username differs
for persona in ['hunter', 'christine']:
    repo_id = f'{HF_USER}/tlc-gemma-4-e4b-{persona}-lora'
    api.create_repo(repo_id, private=False, exist_ok=True)
    api.upload_folder(
        folder_path=f'out/lora-{persona}',
        repo_id=repo_id,
        repo_type='model',
    )
    print(f'pushed {repo_id}')

**B) Download as zip** (one-shot, watch out for the Colab disconnect timer):

In [ ]:
!zip -r out/lora-hunter.zip out/lora-hunter
!zip -r out/lora-christine.zip out/lora-christine
from google.colab import files
files.download('out/lora-hunter.zip')
files.download('out/lora-christine.zip')

## 9. Next: merge + GGUF export

Run `training/merge_and_quantize.sh` on Sam's box (where llama.cpp is built)
to merge each LoRA into the base, convert to GGUF, and quantize to Q5_K_M.
Drop the resulting GGUFs in `~/models/` and point the worker at them.